# Notebook 11 — V3 Biometric Alignment & Embedding Validation

**Single `Runtime → Run all`** in Colab Python 3.12. Uses *frozen* V3 (320px content centred on 640px black canvas) from N10, on the 120 existing original AffectNet+ images. The 14 N08–N10 diagnostic originals are **development**; the remaining 106 are the **held-out diagnostic validation subset**. All 120 are evaluated, but coverage is reported separately.

Uses SHA256-verified N09 detector and ArcFace/AuraFace recognizers, generates 112×112 aligned image previews, tests finite embeddings and geometric consistency, and saves row-level outcomes. **No 600-generated-image evaluation, no cosine similarity or privacy claims.** Manual face correctness stays `pending`, and no model setting is tuned using validation outcomes.

Source images remain in Drive; no prior output is overwritten. `Run all` is unattended after normal Drive authorization. CPU processing; note that Google Colab can still disconnect.

## 1 — Frozen provenance and methods

The V3 method is selected from N10 once; it is **not** optimized again using the 106 additional originals. Freeze N10's 14 image checksums and original pinned packages. The original 120-image source manifest is read from the preserved N07 run.

In [ ]:
import json, hashlib, os, sys, platform, subprocess, tempfile, time, uuid, csv
from pathlib import Path
from datetime import datetime, timezone
CONFIG = json.loads("{\n  \"schema_version\": 1,\n  \"method_id\": \"nb10_detector_scale_canvas_v1\",\n  \"classification\": \"exploratory_scientific_validity_diagnostic\",\n  \"parent07\": \"20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664\",\n  \"parent08\": \"20261013_nb08_biometric-diagnostic_48369686_3f72a02e_20261013T181313795949Z_eb90117f\",\n  \"sources\": [\n    {\n      \"source_id\": \"source_000\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_001\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_005\",\n      \"emotion\": \"happy\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_008\",\n      \"emotion\": \"happy\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_013\",\n      \"emotion\": \"sad\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_010\",\n      \"emotion\": \"sad\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_017\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_015\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_021\",\n      \"emotion\": \"fear\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_020\",\n      \"emotion\": \"fear\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_026\",\n      \"emotion\": \"angry\",\n      \"historical_stratum\": \"success\"\n    },\n    {\n      \"source_id\": \"source_025\",\n      \"emotion\": \"angry\",\n      \"historical_stratum\": \"faceanalysis_failed\"\n    },\n    {\n      \"source_id\": \"source_044\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"yolo_failed\"\n    },\n    {\n      \"source_id\": \"source_019\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"yolo_failed\"\n    }\n  ],\n  \"priority\": [\n    \"source_008\",\n    \"source_015\",\n    \"source_017\",\n    \"source_000\",\n    \"source_013\",\n    \"source_005\",\n    \"source_021\",\n    \"source_026\"\n  ],\n  \"references\": {\n    \"R1\": \"Recomputed N08 A: RGB ndarray YOLO largest box/margin15; compare saved prepared hash; FaceAnalysis 1.0.1 det1024 thresh0.1\",\n    \"R2\": \"Minimal upstream FaceAnalysis full RGB prepared image -> BGR; same models, det1024 thresh0.1; direct detect/get/trace comparison\",\n    \"R3\": \"Independent YOLO face graph on full PIL RGB input, confidence0.25; diagnostic only\",\n    \"R4\": \"Conditional legacy 0.7.3 decoder replay of identical captured raw tensors only if independent decode disagrees with native coordinates\"\n  },\n  \"det_size\": [\n    1024,\n    1024\n  ],\n  \"det_thresh\": 0.1,\n  \"yolo_confidences\": [\n    0.25,\n    0.1,\n    0.05\n  ],\n  \"crop_margin\": 0.15,\n  \"seed\": 42,\n  \"providers\": [\n    \"CPUExecutionProvider\"\n  ],\n  \"device\": \"cpu\",\n  \"models\": {\n    \"arcface\": {\n      \"pack\": \"buffalo_l\",\n      \"archive_url\": \"https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip\",\n      \"onnx_sha256\": {\n        \"1k3d68.onnx\": \"df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\",\n        \"2d106det.onnx\": \"f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\",\n        \"det_10g.onnx\": \"5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\",\n        \"genderage.onnx\": \"4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\",\n        \"w600k_r50.onnx\": \"4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43\"\n      }\n    },\n    \"auraface\": {\n      \"repo\": \"fal/AuraFace-v1\",\n      \"revision\": \"686774cad65e40933022896195e07e01ee06bee9\",\n      \"onnx_sha256\": {\n        \"1k3d68.onnx\": \"df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\",\n        \"2d106det.onnx\": \"f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\",\n        \"genderage.onnx\": \"4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\",\n        \"glintr100.onnx\": \"a7933ea5330113b01c9b60351d8f4c33003f145d8470ac5f0e52ee2effe25c60\",\n        \"scrfd_10g_bnkps.onnx\": \"5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\"\n      }\n    },\n    \"yolo\": {\n      \"repo\": \"Bingsu/adetailer\",\n      \"revision\": \"53cc19de382014514d9d4038601d261a7faa9b7b\",\n      \"filename\": \"face_yolov8n.pt\",\n      \"sha256\": \"70b640f8f60b1cf0dcc72f30caf3da9495eb2fb6509da48c53374ad6806e6a9c\"\n    }\n  },\n  \"pins\": {\n    \"numpy\": \"1.26.4\",\n    \"pandas\": \"2.2.2\",\n    \"Pillow\": \"11.3.0\",\n    \"matplotlib\": \"3.10.0\",\n    \"onnx\": \"1.17.0\",\n    \"onnxruntime\": \"1.19.2\",\n    \"insightface\": \"1.0.1\",\n    \"ultralytics\": \"8.4.121\",\n    \"opencv-python\": \"4.11.0.86\",\n    \"huggingface-hub\": \"0.36.2\",\n    \"scipy\": \"1.15.3\",\n    \"scikit-image\": \"0.25.2\",\n    \"packaging\": \"25.0\",\n    \"torch\": \"2.11.0+cpu\",\n    \"torchvision\": \"0.26.0+cpu\"\n  },\n  \"environment_boundary\": \"CPU-only private environment; same torch/torchvision release as N08, CPU builds replace cu128. YOLO bitwise GPU reproduction not presumed; saved crop/detection comparison reported.\",\n  \"package_source\": {\n    \"wheel_url\": \"https://files.pythonhosted.org/packages/42/ce/33989427a7fae952e1606d0edee4191f965630f3c74f2166fa776fafd271/insightface-1.0.1-py3-none-any.whl\",\n    \"wheel_sha256\": \"5f373f6fedbdda5cbc59a34ca386a75a2995cdaf6899402590ae9eb4308fc2e8\",\n    \"legacy_url\": \"https://files.pythonhosted.org/packages/0b/8d/0f4af90999ca96cf8cb846eb5ae27c5ef5b390f9c090dd19e4fa76364c13/insightface-0.7.3.tar.gz\",\n    \"legacy_sha256\": \"f191f719612ebb37018f41936814500544cd0f86e6fcd676c023f354c668ddf7\",\n    \"legacy_retinaface_sha256\": \"945597b54141a892419e0edfba464eb84fa2ec10ba800265c06ee274edc534ee\",\n    \"cpu_wheels\": {\n      \"torch\": {\n        \"url\": \"https://download-r2.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl\",\n        \"sha256\": \"f82e2ae20c1545bb03997d1cc3143d94e14b800038669ee1aca45808a9acc338\"\n      },\n      \"torchvision\": {\n        \"url\": \"https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl\",\n        \"sha256\": \"cf547dc0975eb40bc3249be4ccbeb736597d2c3ece305b1c4e5b7a5dd7363567\"\n      }\n    }\n  },\n  \"geometry_rules\": {\n    \"min_interocular_px\": 1,\n    \"landmark_box_tolerance_fraction\": 0.25,\n    \"alignment_condition_max\": 100000000,\n    \"warning_padding_fraction\": 0.5,\n    \"automatic_rules\": \"engineering plausibility only; no intended-face acceptance without actual manual review\"\n  },\n  \"privacy_similarity\": \"not_calculated\",\n  \"biometric_threshold\": null,\n  \"prompt\": \"not_applicable_existing_images\",\n  \"mask\": \"none\",\n  \"diffusion_parameters\": \"not_applicable\",\n  \"utility\": \"not_reevaluated\",\n  \"manual_review\": \"pending; never inferred from extraction\",\n  \"cpu_threads\": 2,\n  \"numeric_tolerance\": 0.0001,\n  \"input_contract\": {\n    \"prepared\": \"HWC uint8 BGR 0..255\",\n    \"blob\": \"NCHW float32 RGB (pixel-127.5)/128\",\n    \"pad_value\": 0,\n    \"coordinate_frame\": \"detector.detect returns prepared-input pixel coordinates; crop offsets then map to original\"\n  }\n}")
CONFIG_SHA256 = hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
RUNS_ROOT = Path("/content/drive/MyDrive/thesis_all/outputs/runs")
PARENT07 = RUNS_ROOT / CONFIG["parent07"]
PARENT08 = RUNS_ROOT / CONFIG["parent08"]
WORKER_PARTS = []
print("Frozen configuration:",CONFIG_SHA256,"; 14 sources; CPU diagnostic; no execution switches.")

CONFIG['variants']=[
 {'id':'V0_original_1024','canvas':1024,'kind':'direct','description':'Original full-frame resized by upstream SCRFD to 1024'},
 {'id':'V1_original_640','canvas':640,'kind':'direct','description':'Same source directly, upstream SCRFD resize to 640'},
 {'id':'V2_original_320','canvas':320,'kind':'direct','description':'Same source directly, upstream SCRFD resize to 320'},
 {'id':'V3_centered_640','canvas':640,'kind':'centered','description':'512 original isotropically reduced to 320 and centered on 640 black canvas'},
 {'id':'V4_centered_1024','canvas':1024,'kind':'centered1024','description':'512 original unresized, centered on black 1024 canvas'}]
CONFIG['method_id']='nb10_detector_scale_canvas_v1'
CONFIG['analysis']='detector_localization_only_no_embeddings_or_privacy'
CONFIG_SHA256=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('NB10 frozen methods:', [(x['id'], x['canvas']) for x in CONFIG['variants']]);print('Config hash:',CONFIG_SHA256)

# N11 pre-registration: freeze V3 only, without adaptive choices.
CONFIG['variants']=[{'id':'V3_centered_640','canvas':640,'kind':'centered','content_size':320,'pad_value':0}]
CONFIG['method_id']='nb11_v3_embedding_validation_v1'
CONFIG['analysis']='120_originals_14_development_106_holdout_no_privacy'
CONFIG['det_thresh']=0.1
CONFIG_SHA256=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
print('N11 frozen V3; config:',CONFIG_SHA256)


## 2 — Reuse the verified isolated Python environment

Known Colab 3.12 bootstrap from the successfully executed N10. The `MPLBACKEND=Agg` workaround and strict pins are retained. No manual True/False flags.

In [ ]:

if sys.platform != 'linux' or platform.machine() != 'x86_64' or sys.version_info[:2] != (3,12):
    raise RuntimeError('Notebook 10 needs Linux x86_64 CPython 3.12 (prefer 3.12.13). Select a compatible Colab runtime. No pins are substituted.')
libc_name,libc_version=platform.libc_ver()
if libc_name!='glibc' or tuple(map(int,libc_version.split('.')[:2]))<(2,28):
    raise RuntimeError('Pinned CPU wheels require glibc >=2.28; use compatible Colab Python 3.12 runtime.')
if platform.python_version() != '3.12.13':
    print('Runtime patch differs from N08:',platform.python_version(),'; recorded, never silently equated.')
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')

def sha_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()

PRIVATE_ENV=dict(os.environ)
for name in ['PYTHONPATH','PYTHONHOME','PIP_TARGET','PIP_USER','PIP_PREFIX']:
    PRIVATE_ENV.pop(name,None)
PRIVATE_ENV.update(PIP_CONFIG_FILE=os.devnull,CUDA_VISIBLE_DEVICES='',MPLBACKEND='Agg')

def run_process(command):
    print('Environment step:',command[1:4],flush=True)
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=PRIVATE_ENV)
    for line in process.stdout: print(line.rstrip(),flush=True)
    if process.wait(): raise RuntimeError('Automatic isolated setup failed; no diagnostic inference started. See pip/import error above.')

pin_hash=hashlib.sha256(json.dumps(CONFIG['pins'],sort_keys=True).encode()).hexdigest()[:12]
ENV_DIR=Path('/content')/('nb09_env_'+platform.python_version()+'_'+pin_hash)
ENV_PY=ENV_DIR/'bin'/'python'
PROBE = r'''
import json,sys,platform,hashlib,inspect
from importlib import metadata
import numpy,pandas,PIL,matplotlib,onnx,onnxruntime,insightface,cv2,torch,torchvision,ultralytics
from insightface.app import FaceAnalysis
import insightface.model_zoo.scrfd as scrfd
import insightface.model_zoo.model_zoo as zoo
pins=json.loads(sys.argv[1])
for name,expected in pins.items():
    if metadata.version(name)!=expected: raise RuntimeError(f'Pin mismatch: {name}, expected {expected}')
if numpy.__version__!=pins['numpy'] or cv2.__version__!='.'.join(pins['opencv-python'].split('.')[:3]):
    raise RuntimeError('Native import/version collision')
if 'CPUExecutionProvider' not in onnxruntime.get_available_providers(): raise RuntimeError('CPU ONNX unavailable')
result={'python':platform.python_version(),'platform':platform.platform(),'packages':dict(sorted((d.metadata['Name'],d.version) for d in metadata.distributions())),
 'torch_device':str(torch.device('cpu')),'torch':torch.__version__,'torchvision':torchvision.__version__,
 'onnx_providers':onnxruntime.get_available_providers(),'source_sha256':{}}
for obj in [FaceAnalysis,scrfd.SCRFD,zoo.ModelRouter]:
    path=inspect.getsourcefile(obj)
    result['source_sha256'][obj.__name__]=hashlib.sha256(open(path,'rb').read()).hexdigest()
print('NB09_ENV '+json.dumps(result,sort_keys=True))
'''
# Colab may ship a Python interpreter whose ensurepip bootstrap cannot complete.
# Create a pip-free venv, then bootstrap pinned pip *inside that venv*
# using host pip's --python target (does not install packages into the host).
import venv
import shutil
PIP_BOOTSTRAP_VERSION = '24.3.1'

def private_pip_works():
    if not ENV_PY.is_file():
        return False
    try:
        check = subprocess.run(
            [str(ENV_PY), '-m', 'pip', '--version'],
            env=PRIVATE_ENV, text=True, capture_output=True, timeout=30,
        )
        return check.returncode == 0 and str(ENV_DIR) in check.stdout
    except (OSError, subprocess.TimeoutExpired):
        return False

if not private_pip_works():
    # Only the uniquely named, ephemeral /content/nb09_env_* directory is touched.
    if ENV_DIR.exists():
        if ENV_DIR.parent != Path('/content') or not ENV_DIR.name.startswith('nb09_env_'):
            raise RuntimeError('Unsafe private environment path; refusing cleanup')
        print('Cleaning incomplete Notebook 10 venv:', ENV_DIR, flush=True)
        shutil.rmtree(ENV_DIR)
    try:
        venv.EnvBuilder(with_pip=False, system_site_packages=False).create(ENV_DIR)
    except Exception as error:
        raise RuntimeError('Unable to create pip-free isolated venv: ' + repr(error)) from error
    # pip >=22.3 supports --python and can provision pip into a venv with no pip.
    host_pip = subprocess.run(
        [sys.executable, '-m', 'pip', '--version'],
        env=PRIVATE_ENV, text=True, capture_output=True, timeout=30,
    )
    if host_pip.returncode:
        raise RuntimeError('Host pip unavailable for target-only bootstrap: ' + host_pip.stderr[-3000:])
    run_process([
        sys.executable, '-m', 'pip', '--python', str(ENV_PY),
        '--isolated', 'install', '--disable-pip-version-check',
        '--index-url', 'https://pypi.org/simple',
        'pip==' + PIP_BOOTSTRAP_VERSION,
    ])
    if not private_pip_works():
        raise RuntimeError('Pinned pip bootstrap did not produce working private pip; no inference started')
VERSION_PROBE="import json,sys;from importlib.metadata import version; pins=json.loads(sys.argv[1]);assert all(version(k)==v for k,v in pins.items())"
versions_ready=subprocess.run([str(ENV_PY),'-c',VERSION_PROBE,json.dumps(CONFIG['pins'])],capture_output=True,env=PRIVATE_ENV).returncode==0
if not versions_ready:
    requirements=[name+'=='+version for name,version in CONFIG['pins'].items()]
    requirements=[value for value in requirements if not value.startswith(('insightface==','torch==','torchvision=='))]
    requirements += [spec['url']+'#sha256='+spec['sha256'] for spec in CONFIG['package_source']['cpu_wheels'].values()]
    requirements.append(CONFIG['package_source']['wheel_url']+'#sha256='+CONFIG['package_source']['wheel_sha256'])
    run_process([str(ENV_PY),'-m','pip','--isolated','install','--timeout','30','--retries','2',
                 '--index-url','https://pypi.org/simple',*requirements])
run_process([str(ENV_PY),'-m','pip','--isolated','check'])
probe_env=dict(PRIVATE_ENV,CUDA_VISIBLE_DEVICES='',YOLO_CONFIG_DIR=str(ENV_DIR/'yolo_settings'),MPLCONFIGDIR=str(ENV_DIR/'mpl'))
probe=subprocess.run([str(ENV_PY),'-c',PROBE,json.dumps(CONFIG['pins'])],env=probe_env,text=True,capture_output=True,timeout=180)
if probe.returncode: raise RuntimeError('Private environment import/ABI/pin preflight failed before inference: '+probe.stderr[-6000:])
probe_lines=[line[9:] for line in probe.stdout.splitlines() if line.startswith('NB09_ENV ')]
if len(probe_lines)!=1: raise RuntimeError('Environment probe missing structured output')
ENV_METADATA=json.loads(probe_lines[0])
EXPECTED_PACKAGE_SOURCE={'FaceAnalysis':'759271af722dafa1f602354550cdfd84cd705de0184a63b43752982635bcf5ff',
                         'SCRFD':'8898d1f90690d46bee371e1de31d1af60837988272a5eef904c5ba89e0fb8d83'}
for name,expected in EXPECTED_PACKAGE_SOURCE.items():
    if ENV_METADATA['source_sha256'][name]!=expected:raise RuntimeError('Installed official InsightFace source mismatch: '+name)
ENV_SHA256=hashlib.sha256(json.dumps(ENV_METADATA,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('Isolated environment verified; no restart/model loading. Fingerprint:',ENV_SHA256)


## 3 — Read-only provenance and source preflight

All 120 source PNG files must exist with unique identifiers. The 14 original diagnostic hashes must remain unchanged. All three ONNX weights are copied **by reference only**, with their SHA256 checked in the worker. Output path is unique for each run.

In [ ]:
from PIL import Image
import json as _json
NB09_RUN=RUNS_ROOT/'20261013_nb09_detector-rootcause_4fae269d_3a2979d8_20261013T201108057643Z_02a058fe'
NB09_SESSION=NB09_RUN/'sessions'/'20261013T201108057643Z_02a058fe'
NB09_HASHES=NB09_SESSION/'model_file_hashes.json'
NB09_MANIFEST=NB09_SESSION/'session_manifest.json'
NB10_RUN=RUNS_ROOT/'nb10_scale_canvas_c932b62da2be_20261013T203150121955Z_5366f3bc'
PARENT07=RUNS_ROOT/CONFIG['parent07']
for required in [NB09_HASHES,NB09_MANIFEST,NB10_RUN/'summary.json',NB10_RUN/'config_and_context.json']:
    if not required.is_file():raise FileNotFoundError('Missing historical immutable evidence: '+str(required))
nb10_summary=json.loads((NB10_RUN/'summary.json').read_text())
v3=next(v for v in nb10_summary['variants'] if v['variant']=='V3_centered_640')
if v3['detected']!=14 or v3['errors']!=0:raise RuntimeError('Preserved N10 V3 evidence differs')
archive=json.loads(NB09_HASHES.read_text())
expected={
 'detector':CONFIG['models']['arcface']['onnx_sha256']['det_10g.onnx'],
 'arcface':CONFIG['models']['arcface']['onnx_sha256']['w600k_r50.onnx'],
 'auraface':CONFIG['models']['auraface']['onnx_sha256']['glintr100.onnx'],
}
resolved={}
for name,sha in expected.items():
    candidates=[Path(p) for p,h in archive.items() if h==sha and Path(p).is_file()]
    if not candidates:raise FileNotFoundError('N09 weight unavailable for '+name)
    path=candidates[0]
    if sha_file(path)!=sha:raise RuntimeError('Model hash mismatch: '+name)
    resolved[name]=str(path)
source_dir=PARENT07/'sources'
if not source_dir.is_dir():raise FileNotFoundError('Original 120-image directory missing: '+str(source_dir))
all_paths=sorted(source_dir.glob('source_*.png'))
if len(all_paths)!=120:raise RuntimeError(f'Expected exactly 120 source PNGs; found {len(all_paths)}. Stop rather than silently change study cohort.')
all_sources=[]
for p in all_paths:
    if not p.stem.startswith('source_') or p.stem in {r['source_id'] for r in all_sources}:raise RuntimeError('Duplicate/invalid source ID')
    all_sources.append({'source_id':p.stem,'path':str(p),'sha256':sha_file(p)})
private_manifest=PARENT07/'metadata'/'private_development_source_hashes.json'
if not private_manifest.is_file(): raise RuntimeError('Private verification manifest missing')
expected_hashes=json.loads(private_manifest.read_text())
dev={r['source_id']:r for r in CONFIG['sources']}
assert len(dev)==14
for r in all_sources:
    r['subset']='development_14' if r['source_id'] in dev else 'heldout_106'
    if r['subset']=='development_14' and r['sha256']!=expected_hashes[r['source_id']]:
        raise RuntimeError('Frozen diagnostic source changed: '+r['source_id'])
assert sum(r['subset']=='development_14' for r in all_sources)==14
assert sum(r['subset']=='heldout_106' for r in all_sources)==106
run_id='nb11_v3_alignment_'+CONFIG_SHA256[:12]+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'_'+uuid.uuid4().hex[:8]
OUT_DIR=RUNS_ROOT/run_id
OUT_DIR.mkdir(exist_ok=False)
context={'run_id':run_id,'config':CONFIG,'config_hash':CONFIG_SHA256,'models':resolved,'model_sha256':expected,
         'sources':all_sources,'nb09_hash_manifest_sha256':sha_file(NB09_HASHES),
         'nb10_summary_sha256':sha_file(NB10_RUN/'summary.json'),'manual_review':'pending'}
(OUT_DIR/'context.json').write_text(json.dumps(context,indent=2,allow_nan=False))
print('Prepared',len(all_sources),'originals; dev 14 / heldout 106. Outputs:',OUT_DIR)


## 4 — Automatic independent detector → 5-point alignment → ArcFace/AuraFace embeddings

Only the N10 V3 image preparation is used. The *same* SCRFD 5-point landmarks align a single 112px face crop; the aligned crop is supplied to both recognizers. Strict validations include bbox/landmark geometry, array shape, finite values, embedding dimension and L2 norm. Technical extraction success is **not** assumed to mean the intended face was recognized. Saves annotated detection and crops for later human verification.

The 120 originals are existing research data, not newly generated. Output `records.csv`, 120 source-level JSON files, contact sheets and `summary.json` are written only to N11's new directory.

In [ ]:
WORKER_SOURCE = 'import os,sys,json,hashlib,traceback,csv,time,math\nfrom pathlib import Path\nimport numpy as np, cv2\nfrom PIL import Image,ImageDraw,ImageOps\nfrom insightface.model_zoo import get_model\nfrom insightface.utils import face_align\nctx=json.loads(Path(sys.argv[1]).read_text()); root=Path(sys.argv[2]); cfg=ctx[\'config\']\ndef digest(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\ndef write_json(path,value):\n tmp=path.with_suffix(path.suffix+\'.partial\')\n with open(tmp,\'x\') as f:json.dump(value,f,indent=2,allow_nan=False)\n os.replace(tmp,path)\ndef safe_float(x):\n v=float(x);return v if math.isfinite(v) else None\nfor k,p in ctx[\'models\'].items():\n if digest(p)!=ctx[\'model_sha256\'][k]:raise RuntimeError(\'Verified model hash mismatch \'+k)\nprint(\'Loading N09 verified SCRFD + two recognition models\',flush=True)\ndet=get_model(ctx[\'models\'][\'detector\'],providers=[\'CPUExecutionProvider\'])\nassert type(det).__name__==\'SCRFD\'\ndet.prepare(ctx_id=-1,input_size=(640,640),det_thresh=0.1)\nrec={}\nfor k in (\'arcface\',\'auraface\'):\n model=get_model(ctx[\'models\'][k],providers=[\'CPUExecutionProvider\'])\n model.prepare(ctx_id=-1)\n rec[k]=model\nnp.random.seed(42)\n# Test coordinate math independently of real inference\nT=np.array([[0.625,0,160],[0,0.625,160],[0,0,1.]],dtype=np.float64)\nprobe=np.array([[0.,0.],[512.,512.],[123.,256.]])\nz=np.concatenate([probe,np.ones((len(probe),1))],axis=1)@T.T\nassert np.allclose((z@np.linalg.inv(T).T)[:,:2],probe,atol=1e-7)\nresults=[]\nfor idx,source in enumerate(ctx[\'sources\']):\n sid=source[\'source_id\'];out=root/(sid+\'.json\');sheet=root/(sid+\'_preview.png\')\n if out.exists():\n  data=json.loads(out.read_text())\n  if data.get(\'source_sha256\')!=source[\'sha256\'] or data.get(\'config_hash\')!=ctx[\'config_hash\']:\n   raise RuntimeError(\'Unsafe checkpoint mismatch \'+sid)\n  if not sheet.exists():raise RuntimeError(\'Partial image checkpoint \'+sid)\n  results.append(data);continue\n start=time.perf_counter()\n data={\'source_id\':sid,\'subset\':source[\'subset\'],\'source_sha256\':source[\'sha256\'],\'config_hash\':ctx[\'config_hash\'],\n       \'preprocessing\':\'V3: 512px source to 320px image centered in 640 black canvas, 0.1 threshold\',\n       \'num_detections\':0,\'selected_bbox_source\':None,\'selected_landmarks_source\':None,\'detection_confidence\':None,\n       \'box_intersection_fraction\':None,\'landmarks_plausible\':False,\'alignment_plausible\':False,\n       \'recognizers\':{},\'manual_face_acceptance\':\'pending\',\'error\':None}\n try:\n  if digest(source[\'path\'])!=source[\'sha256\']:raise RuntimeError(\'Source hash changed\')\n  rgb=np.asarray(Image.open(source[\'path\']).convert(\'RGB\'))\n  h,w=rgb.shape[:2]\n  if (h,w)!=(512,512):raise RuntimeError(\'Unexpected original source shape\')\n  bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)\n  centered=np.zeros((640,640,3),dtype=np.uint8)\n  centered[160:480,160:480]=cv2.resize(bgr,(320,320),interpolation=cv2.INTER_AREA)\n  boxes,landmarks=det.detect(centered,max_num=0,metric=\'default\')\n  data[\'num_detections\']=len(boxes)\n  preview=Image.new(\'RGB\',(768,384),(250,250,250))\n  original_img=Image.fromarray(rgb).resize((384,384))\n  draw=ImageDraw.Draw(original_img)\n  aligned=None\n  if len(boxes):\n   b=boxes[0];pts=np.asarray(landmarks[0],dtype=np.float64)\n   box=((np.asarray(b[:4],dtype=np.float64)-160)/0.625).tolist()\n   kps=((pts-160)/0.625).tolist()\n   data[\'selected_bbox_source\']=[safe_float(v) for v in box]\n   data[\'selected_landmarks_source\']=[[safe_float(v) for v in xy] for xy in kps]\n   data[\'detection_confidence\']=safe_float(b[4])\n   l,t,r,bottom=box\n   intersection=max(0,min(r,w)-max(l,0))*max(0,min(bottom,h)-max(t,0))\n   area=max(0,r-l)*max(0,bottom-t)\n   data[\'box_intersection_fraction\']=safe_float(intersection/area) if area else 0.\n   eye_dist=float(np.linalg.norm(np.asarray(kps[0])-np.asarray(kps[1])))\n   data[\'landmarks_plausible\']=bool(np.isfinite(pts).all() and pts.shape==(5,2) and eye_dist>=1.0\n       and l<r and t<bottom and data[\'box_intersection_fraction\']>=0.7)\n   # Do not gate extraction on engineering heuristics: report actual extractor behavior separately.\n   try:\n    aligned=face_align.norm_crop(bgr,landmark=np.asarray(kps,dtype=np.float32),image_size=112)\n    data[\'alignment_plausible\']=bool(aligned.shape==(112,112,3) and aligned.dtype==np.uint8\n       and np.isfinite(aligned).all() and data[\'landmarks_plausible\'])\n   except Exception as e:\n    data[\'alignment_error\']=type(e).__name__+\': \'+str(e)\n   draw.rectangle(tuple(float(v)*0.75 for v in box),outline=\'yellow\',width=3)\n   for x,y in kps:\n    draw.ellipse(((x-3)*.75,(y-3)*.75,(x+3)*.75,(y+3)*.75),fill=\'red\')\n  preview.paste(original_img,(0,0))\n  if aligned is not None:\n   crop_rgb=cv2.cvtColor(aligned,cv2.COLOR_BGR2RGB)\n   crop_img=Image.fromarray(crop_rgb).resize((336,336),Image.Resampling.NEAREST)\n   preview.paste(crop_img,(408,24))\n   for name,recognizer in rec.items():\n    try:\n     emb=np.asarray(recognizer.get_feat(aligned),dtype=np.float32).reshape(-1)\n     norm=float(np.linalg.norm(emb))\n     valid=bool(emb.size==512 and np.isfinite(emb).all() and np.isfinite(norm) and norm>1e-10)\n     data[\'recognizers\'][name]={\'embedding_extracted\':valid,\'dim\':int(emb.size),\'norm\':safe_float(norm),\n       \'embedding_sha256\':hashlib.sha256(emb.tobytes()).hexdigest() if valid else None}\n    except Exception as e:\n     data[\'recognizers\'][name]={\'embedding_extracted\':False,\'error\':type(e).__name__+\': \'+str(e)}\n  else:\n   for name in rec:data[\'recognizers\'][name]={\'embedding_extracted\':False,\'reason\':\'no_detection_or_alignment\'}\n  data[\'elapsed_seconds\']=time.perf_counter()-start\n  preview.save(sheet)\n except Exception as e:\n  data[\'error\']=type(e).__name__+\': \'+str(e)\n  data[\'traceback\']=traceback.format_exc()\n  data[\'elapsed_seconds\']=time.perf_counter()-start\n  # Persist a diagnostic image even for failed cases.\n  Image.new(\'RGB\',(768,384),\'white\').save(sheet)\n write_json(out,data)\n results.append(data)\n if (idx+1)%10==0 or idx==0:print(f\'{idx+1}/{len(ctx["sources"])} sources; detections={data["num_detections"]}; error={data["error"]}\',flush=True)\n# Write the table only after source-level immutable checkpoints are complete.\ncolumns=[\'source_id\',\'subset\',\'num_detections\',\'detection_confidence\',\'box_intersection_fraction\',\'landmarks_plausible\',\'alignment_plausible\',\n         \'arcface_embedding_extracted\',\'auraface_embedding_extracted\',\'manual_face_acceptance\',\'error\']\nwith open(root/\'records.csv\',\'x\',newline=\'\') as f:\n writer=csv.DictWriter(f,fieldnames=columns);writer.writeheader()\n for r in results:\n  writer.writerow({**{k:r.get(k) for k in columns},\n      \'arcface_embedding_extracted\':r.get(\'recognizers\',{}).get(\'arcface\',{}).get(\'embedding_extracted\',False),\n      \'auraface_embedding_extracted\':r.get(\'recognizers\',{}).get(\'auraface\',{}).get(\'embedding_extracted\',False)})\nsummary={\'status\':\'completed_pending_manual_review\',\'run_id\':ctx[\'run_id\'],\'scientific_config_sha256\':ctx[\'config_hash\'],\n         \'records\':len(results),\'manual_face_acceptance\':\'pending\',\'face_assignment_validated_count\':None,\n         \'biometric_coverage_validated\':None,\'privacy_similarity\':\'not_calculated\',\'subsets\':{}}\nfor group in (\'development_14\',\'heldout_106\',\'all_120\'):\n rrs=results if group==\'all_120\' else [r for r in results if r[\'subset\']==group]\n summary[\'subsets\'][group]={\'n\':len(rrs),\'detection_success\':sum(x[\'num_detections\']>0 for x in rrs),\n   \'geometric_plausibility\':sum(bool(x[\'alignment_plausible\']) for x in rrs),\n   \'arcface_embedding_extraction\':sum(bool(x.get(\'recognizers\',{}).get(\'arcface\',{}).get(\'embedding_extracted\')) for x in rrs),\n   \'auraface_embedding_extraction\':sum(bool(x.get(\'recognizers\',{}).get(\'auraface\',{}).get(\'embedding_extracted\')) for x in rrs),\n   \'technical_errors\':sum(x[\'error\'] is not None for x in rrs)}\nwrite_json(root/\'summary.json\',summary)\n# Contact sheets use saved previews only, images scaled down for review navigation.\nfor group in (\'development_14\',\'heldout_106\'):\n rr=[r for r in results if r[\'subset\']==group]\n for page,start in enumerate(range(0,len(rr),20),1):\n  batch=rr[start:start+20]\n  sheet=Image.new(\'RGB\',(4*400,5*230),\'white\');d=ImageDraw.Draw(sheet)\n  for i,r in enumerate(batch):\n   path=root/(r[\'source_id\']+\'_preview.png\')\n   im=Image.open(path).convert(\'RGB\').resize((392,196))\n   col=i%4;row=i//4;xx=col*400;yy=row*230\n   sheet.paste(im,(xx,yy+20));d.text((xx+4,yy+2),f"{r[\'source_id\']}  detected={r[\'num_detections\']}  align={r[\'alignment_plausible\']}",fill=\'black\')\n  sheet.save(root/(f\'contact_{group}_page_{page:02d}.png\'))\nlines=[\'# Notebook 11 — V3 biometric alignment validation\',\'\',f"Completed {len(results)}/120 originals; frozen V3.",\'\',\n \'Group | N | Detected | Geometry/alignment plausible | ArcFace finite embeddings | AuraFace finite embeddings | Technical errors\',\n \'|---|---:|---:|---:|---:|---:|---:|\']\nfor key,v in summary[\'subsets\'].items():\n lines.append(f"{key} | {v[\'n\']} | {v[\'detection_success\']} | {v[\'geometric_plausibility\']} | {v[\'arcface_embedding_extraction\']} | {v[\'auraface_embedding_extraction\']} | {v[\'technical_errors\']}")\nlines+=[\'\',\'**This is extraction and engineering geometry coverage, NOT validated biometric coverage.**\',\n \'**All intended-face, recognition quality and alignment acceptance decisions remain pending human visual review.**\',\n \'N10 V3 was selected on 14 development images; results on the additional 106 are held-out from that detector-scale selection.\',\n \'No original or generated images were modified. No privacy similarity or re-identification metric was computed.\',\n \'If embedding extraction fails, inspect per-source JSON errors, aligned previews and model-contract assumptions.\']\n(root/\'diagnostic_report.md\').write_text(\'\\n\'.join(lines))\nprint(\'NB11_COMPLETED \'+str(root),flush=True)\n'
WORKER_PATH=OUT_DIR/'worker.py'
WORKER_PATH.write_text(WORKER_SOURCE)
process=subprocess.Popen([str(ENV_PY),'-u',str(WORKER_PATH),str(OUT_DIR/'context.json'),str(OUT_DIR)],
 stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=probe_env)
for line in process.stdout:print(line.rstrip(),flush=True)
if process.wait()!=0:raise RuntimeError('NB11 worker failed; inspect latest log above. Partial source checkpoints saved in fresh run directory.')


## 5 — Evidence report and decision boundary

Only after a person has reviewed intended faces in the saved contact sheets can the protocol be considered for a *separate* final 600-output privacy re-evaluation. Report held-out performance without imputing missing records; never reinterpret `embedding_extracted` as identity-recognition accuracy.

In [ ]:
summary=json.loads((OUT_DIR/'summary.json').read_text())
print((OUT_DIR/'diagnostic_report.md').read_text())
print('OUTPUT FOLDER:',OUT_DIR)
assert summary['records']==120
assert summary['subsets']['heldout_106']['n']==106
assert summary['face_assignment_validated_count'] is None
